# CIFAR-10 Image Classification: Frozen Backbone vs Fine-Tuning
 
**Models compared:** ResNet-18 with a frozen backbone vs ResNet-18 with `layer4` fine-tuning

## Objective
Compare two transfer-learning strategies on CIFAR-10:

1. **Frozen backbone:** train only the final classification layer.
2. **Fine-tuning:** train the final classification layer and the last ResNet block (`layer4`).

Both approaches use the same train/validation/test split. Model selection uses validation metrics; the held-out test set is evaluated after training.


## 1. Install dependencies (run only if needed)

If the libraries are already installed, skip this cell. If installation fails with an SSL certificate error on Windows/Anaconda, use the troubleshooting steps in the README rather than repeatedly running multiple pip commands.

In [ ]:
%pip install torch torchvision numpy pandas matplotlib scikit-learn pillow

## 2. Imports and configuration

In [ ]:
import copy
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from torchvision.models import ResNet18_Weights

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    ConfusionMatrixDisplay,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Quick mode is for checking the pipeline. Use full mode for portfolio metrics.
QUICK_RUN = True
EPOCHS = 2 if QUICK_RUN else 8
BATCH_SIZE = 64
IMAGE_SIZE = 96
NUM_WORKERS = 0  # Safe default for Windows/Jupyter.
HEAD_LR = 1e-3
LAYER4_LR = 1e-4
WEIGHT_DECAY = 1e-4

DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

print(f"Device: {DEVICE}")
print(f"Quick run: {QUICK_RUN}")
print(f"Epochs per strategy: {EPOCHS}")

## 3. Load and inspect CIFAR-10
The official training set is split into training and validation subsets with stratification. The official test set is held out for final evaluation.

In [ ]:
normalization = transforms.Normalize(
    mean=(0.485, 0.456, 0.406),
    std=(0.229, 0.224, 0.225),
)

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    normalization,
])

train_full_eval = datasets.CIFAR10(
    root=str(DATA_DIR), train=True, download=True, transform=eval_transform
)
test_full = datasets.CIFAR10(
    root=str(DATA_DIR), train=False, download=True, transform=eval_transform
)

print(f"Official training examples: {len(train_full_eval):,}")
print(f"Official test examples: {len(test_full):,}")
print(f"Classes: {class_names}")

raw_train = datasets.CIFAR10(
    root=str(DATA_DIR), train=True, download=False, transform=None
)
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for class_id, ax in enumerate(axes.flat):
    idx = next(i for i, label in enumerate(raw_train.targets) if label == class_id)
    image, label = raw_train[idx]
    ax.imshow(image)
    ax.set_title(class_names[label])
    ax.axis("off")
plt.suptitle("Example images from CIFAR-10")
plt.tight_layout()
plt.show()

## 4. Create identical data splits for both experiments
Training images use random horizontal flips. Validation and test transformations are deterministic. Quick mode uses small reproducible subsets to reduce runtime.

In [ ]:
all_indices = np.arange(len(train_full_eval))
all_labels = np.asarray(train_full_eval.targets)

train_indices, val_indices = train_test_split(
    all_indices,
    test_size=0.1,
    random_state=SEED,
    stratify=all_labels,
)

if QUICK_RUN:
    rng = np.random.default_rng(SEED)
    train_indices = rng.choice(train_indices, size=min(3000, len(train_indices)), replace=False)
    val_indices = rng.choice(val_indices, size=min(600, len(val_indices)), replace=False)
    test_indices = rng.choice(np.arange(len(test_full)), size=min(1000, len(test_full)), replace=False)
else:
    test_indices = np.arange(len(test_full))

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    normalization,
])

train_full_augmented = datasets.CIFAR10(
    root=str(DATA_DIR), train=True, download=False, transform=train_transform
)

train_dataset = Subset(train_full_augmented, train_indices.tolist())
val_dataset = Subset(train_full_eval, val_indices.tolist())
test_dataset = Subset(test_full, test_indices.tolist())

pin_memory = DEVICE.type == "cuda"
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=pin_memory
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=pin_memory
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=pin_memory
)

print(f"Training examples: {len(train_dataset):,}")
print(f"Validation examples: {len(val_dataset):,}")
print(f"Test examples: {len(test_dataset):,}")

## 5. Build the two transfer-learning strategies

- **Frozen backbone:** all pretrained layers remain frozen; only `fc` is trained.
- **Fine-tuned layer4:** `layer4` and `fc` are trainable. Earlier feature layers stay frozen.

A lower learning rate is used for `layer4` because it contains pretrained features that should be adjusted carefully. Frozen BatchNorm layers are kept in evaluation mode so their running statistics do not drift while the layers are frozen.

In [ ]:
def build_model(strategy, pretrained_weights=True):
    if strategy not in {"frozen", "fine_tune_layer4"}:
        raise ValueError("strategy must be 'frozen' or 'fine_tune_layer4'")

    weights = ResNet18_Weights.DEFAULT if pretrained_weights else None
    model = models.resnet18(weights=weights)

    # Freeze the full pretrained backbone first.
    for parameter in model.parameters():
        parameter.requires_grad = False

    # Replace the original 1000-class ImageNet head with a 10-class head.
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, len(class_names))

    if strategy == "fine_tune_layer4":
        for parameter in model.layer4.parameters():
            parameter.requires_grad = True

    model = model.to(DEVICE)

    if strategy == "frozen":
        optimizer = torch.optim.Adam(
            model.fc.parameters(), lr=HEAD_LR, weight_decay=WEIGHT_DECAY
        )
    else:
        optimizer = torch.optim.Adam(
            [
                {"params": model.layer4.parameters(), "lr": LAYER4_LR},
                {"params": model.fc.parameters(), "lr": HEAD_LR},
            ],
            weight_decay=WEIGHT_DECAY,
        )

    trainable_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_count = sum(p.numel() for p in model.parameters())
    print(
        f"{strategy}: trainable {trainable_count:,} / {total_count:,} parameters "
        f"({trainable_count / total_count:.2%})"
    )
    return model, optimizer

criterion = nn.CrossEntropyLoss()

## 6. Training and validation helpers
The best checkpoint for each strategy is selected using the lowest validation loss. Training time is measured for a fair, practical comparison on the same machine.

In [ ]:
def set_training_mode(model, strategy):
    model.train()
    if strategy == "frozen":
        # Keep all pretrained layers (including BatchNorm) in eval mode.
        model.eval()
        model.fc.train()
    else:
        # Earlier frozen layers stay in eval mode; layer4 and fc train.
        model.eval()
        model.layer4.train()
        model.fc.train()


def run_epoch(model, loader, criterion, optimizer=None, strategy="frozen"):
    is_training = optimizer is not None
    if is_training:
        set_training_mode(model, strategy)
    else:
        model.eval()

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        if is_training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_training):
            logits = model(images)
            loss = criterion(logits, labels)
            if is_training:
                loss.backward()
                optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_examples += batch_size

    return total_loss / total_examples, total_correct / total_examples


def train_strategy(strategy):
    model, optimizer = build_model(strategy, pretrained_weights=True)
    history = []
    best_val_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    start_total = time.perf_counter()

    checkpoint_path = OUTPUT_DIR / f"best_cifar10_resnet18_{strategy}.pth"

    for epoch in range(EPOCHS):
        epoch_start = time.perf_counter()
        train_loss, train_acc = run_epoch(
            model, train_loader, criterion, optimizer=optimizer, strategy=strategy
        )
        val_loss, val_acc = run_epoch(
            model, val_loader, criterion, optimizer=None, strategy=strategy
        )
        seconds = time.perf_counter() - epoch_start

        history.append({
            "strategy": strategy,
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "train_accuracy": train_acc,
            "val_accuracy": val_acc,
            "seconds": seconds,
        })

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, checkpoint_path)

        print(
            f"{strategy} | Epoch {epoch + 1}/{EPOCHS} | "
            f"train loss {train_loss:.4f}, acc {train_acc:.3%} | "
            f"val loss {val_loss:.4f}, acc {val_acc:.3%} | {seconds:.1f}s"
        )

    model.load_state_dict(best_state)
    elapsed = time.perf_counter() - start_total
    history_df = pd.DataFrame(history)
    return {
        "strategy": strategy,
        "model": model,
        "history": history_df,
        "best_val_loss": float(history_df["val_loss"].min()),
        "best_val_accuracy": float(
            history_df.loc[history_df["val_loss"].idxmin(), "val_accuracy"]
        ),
        "training_seconds": elapsed,
        "checkpoint": str(checkpoint_path),
    }

## 7. Run both experiments
This is the main comparison. Both experiments use the same data split and number of epochs. The pretrained weights are loaded for each strategy.

In [ ]:
results = {}
all_history = []

for strategy in ["frozen", "fine_tune_layer4"]:
    print("\n" + "=" * 72)
    print(f"Training strategy: {strategy}")
    result = train_strategy(strategy)
    results[strategy] = result
    all_history.append(result["history"])

history_df = pd.concat(all_history, ignore_index=True)
history_df.to_csv(OUTPUT_DIR / "comparison_training_history.csv", index=False)

display(history_df.round(4))

## 8. Compare training curves
Compare validation loss and accuracy across epochs. A short quick run is useful for checking the pipeline but may not be enough for a stable conclusion.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for strategy, group in history_df.groupby("strategy"):
    axes[0].plot(group["epoch"], group["val_loss"], marker="o", label=strategy)
    axes[1].plot(group["epoch"], group["val_accuracy"], marker="o", label=strategy)

axes[0].set_title("Validation loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-entropy loss")
axes[0].legend()

axes[1].set_title("Validation accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()

## 9. Evaluate both models on the held-out test set
The test set is not used for training or model selection. The metrics below include accuracy and macro-averaged precision, recall, and F1-score.

In [ ]:
def evaluate_model(model, loader):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=True)
            logits = model(images)
            predictions = logits.argmax(dim=1).cpu().numpy().tolist()
            y_pred.extend(predictions)
            y_true.extend(labels.numpy().tolist())

    accuracy = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    return {
        "accuracy": float(accuracy),
        "macro_precision": float(precision),
        "macro_recall": float(recall),
        "macro_f1": float(f1),
        "y_true": y_true,
        "y_pred": y_pred,
    }

comparison_rows = []
test_results = {}

for strategy, result in results.items():
    metrics = evaluate_model(result["model"], test_loader)
    test_results[strategy] = metrics
    comparison_rows.append({
        "strategy": strategy,
        "best_validation_loss": result["best_val_loss"],
        "best_validation_accuracy": result["best_val_accuracy"],
        "test_accuracy": metrics["accuracy"],
        "test_macro_precision": metrics["macro_precision"],
        "test_macro_recall": metrics["macro_recall"],
        "test_macro_f1": metrics["macro_f1"],
        "training_seconds": result["training_seconds"],
        "trainable_parameters": sum(
            p.numel() for p in result["model"].parameters() if p.requires_grad
        ),
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values(
    by="best_validation_loss", ascending=True
).reset_index(drop=True)

display(comparison_df.round(4))
comparison_df.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)

## 10. Classification report and confusion matrix for the validation-selected model
For a single final model, choose the strategy with the best validation loss. The test metrics above show how both strategies performed on the held-out set.

In [ ]:
selected_strategy = min(
    results.keys(), key=lambda name: results[name]["best_val_loss"]
)
selected_model = results[selected_strategy]["model"]
selected_metrics = test_results[selected_strategy]

print(f"Selected using validation loss: {selected_strategy}")
print(f"Test accuracy: {selected_metrics['accuracy']:.3%}")
print()
print(classification_report(
    selected_metrics["y_true"],
    selected_metrics["y_pred"],
    labels=list(range(10)),
    target_names=class_names,
    zero_division=0,
))

fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay.from_predictions(
    selected_metrics["y_true"],
    selected_metrics["y_pred"],
    labels=list(range(10)),
    display_labels=class_names,
    xticks_rotation=45,
    cmap="Blues",
    ax=ax,
    colorbar=False,
)
ax.set_title(f"Confusion Matrix — {selected_strategy}")
plt.tight_layout()
plt.show()

torch.save(
    selected_model.state_dict(),
    OUTPUT_DIR / f"selected_model_{selected_strategy}.pth"
)
print(f"Selected checkpoint: {OUTPUT_DIR / f'selected_model_{selected_strategy}.pth'}")

## 11. Inspect example predictions

In [ ]:
sample_loader = DataLoader(
    test_dataset, batch_size=1, shuffle=False, num_workers=NUM_WORKERS
)
sample_items = []
selected_model.eval()

with torch.no_grad():
    for image_tensor, label in sample_loader:
        image_tensor = image_tensor.to(DEVICE)
        logits = selected_model(image_tensor)
        probs = torch.softmax(logits, dim=1)[0]
        pred = int(logits.argmax(dim=1).item())
        sample_items.append((
            image_tensor.cpu()[0], int(label.item()), pred, float(probs[pred].item())
        ))
        if len(sample_items) >= 12:
            break

mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for ax, (tensor_img, true_label, pred_label, confidence) in zip(axes.flat, sample_items):
    image = (tensor_img * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
    correct = true_label == pred_label
    ax.imshow(image)
    ax.set_title(
        f"True: {class_names[true_label]}\nPred: {class_names[pred_label]} ({confidence:.0%})",
        color="green" if correct else "red",
        fontsize=9,
    )
    ax.axis("off")
plt.tight_layout()
plt.show()

## 12. Predict a custom image
Set `IMAGE_PATH` to a local image. The model always returns one of the ten CIFAR-10 classes, even if the image is outside those classes; confidence is not proof that the image belongs to CIFAR-10.

In [ ]:
def predict_image(image_path, model=selected_model, top_k=5):
    image = Image.open(image_path).convert("RGB")
    input_tensor = eval_transform(image).unsqueeze(0).to(DEVICE)

    model.eval()
    with torch.no_grad():
        probabilities = torch.softmax(model(input_tensor), dim=1)[0].cpu().numpy()

    ranking = np.argsort(probabilities)[::-1][:top_k]
    return pd.DataFrame({
        "class": [class_names[i] for i in ranking],
        "probability": [float(probabilities[i]) for i in ranking],
    })

# Example:
# display(predict_image("path/to/image.jpg"))
print("Set an image path and call predict_image(path) to test a custom image.")

## 13. Save experiment summary

In [ ]:
summary = {
    "selected_strategy_by_validation_loss": selected_strategy,
    "quick_run": QUICK_RUN,
    "epochs_per_strategy": EPOCHS,
    "train_examples": len(train_dataset),
    "validation_examples": len(val_dataset),
    "test_examples": len(test_dataset),
    "device": str(DEVICE),
}
summary_df = pd.DataFrame([summary])
display(summary_df.T.rename(columns={0: "Value"}))
summary_df.to_csv(OUTPUT_DIR / "experiment_summary.csv", index=False)

print("Saved outputs:")
for path in sorted(OUTPUT_DIR.glob("*")):
    print(" -", path)

## 14. Conclusions and limitations


### What this project demonstrates
- Transfer learning for ten-class image classification.
- A controlled comparison between a frozen feature extractor and partial fine-tuning.
- Stratified train/validation split and a held-out test set.
- Evaluation using accuracy, macro precision/recall/F1, classification report, and confusion matrix.
- Reproducible experiment outputs, model checkpoints, and timing comparison.

### Limitations
- Quick mode uses small subsets and only two epochs; results are for pipeline verification, not final benchmarking.
- CIFAR-10 images are low-resolution and limited to ten classes.
- Results can vary by hardware, library versions, and GPU determinism.
- Fine-tuning may not always improve performance; compare the measured results.

